# 04 · 多工具與工具路由

真實 agent 動輒十幾支工具、還常常要新增。這課把工具系統升級成**註冊表（registry）**：工具即資料，prompt 自動生成；並處理「模型亂呼叫」的健壯性。

## 1. 載入模型與基礎工具

In [ ]:
%pip install -q -U "transformers>=4.45" accelerate bitsandbytes

In [ ]:
import json, re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"   # 想更強可換 Qwen2.5-3B-Instruct，T4 也裝得下

_bnb = BitsAndBytesConfig(
    load_in_4bit=True,                    # 4-bit 量化：模型體積砍約 75%
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=_bnb, device_map="auto", torch_dtype=torch.float16,
)
model.eval()
print(f"已載入 {MODEL_ID}（4-bit）於 {model.device}")


@torch.no_grad()
def chat(messages, max_new_tokens=512, temperature=0.0):
    """LLM 的唯一介面：丟一串 messages，回模型吐的純文字。

    messages = [{"role": "system"|"user"|"assistant", "content": "..."}]
    temperature=0 → 貪婪解碼（穩定、可重現），>0 → 取樣（多樣）。
    整條軌道只透過這個函式碰模型；要換成別的模型/API，改這裡就好。
    """
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    gen = dict(max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    if temperature > 0:
        gen.update(do_sample=True, temperature=temperature, top_p=0.9)
    else:
        gen.update(do_sample=False)
    out = model.generate(**inputs, **gen)
    reply = out[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(reply, skip_special_tokens=True).strip()

In [ ]:
from datetime import datetime, timezone, timedelta


def get_current_time() -> str:
    """回傳台北現在時間（模型沒有時鐘，這是它拿不到的真實世界資訊）。"""
    tz = timezone(timedelta(hours=8))
    return datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")


def calculator(expression: str) -> str:
    """安全地算一個算式。只允許數字與 + - * / ( ) . %。"""
    if not re.fullmatch(r"[0-9+\-*/(). %]+", expression or ""):
        return "錯誤：只接受數字與 + - * / ( ) . %"
    try:
        return str(eval(expression, {"__builtins__": {}}, {}))
    except Exception as e:
        return f"錯誤：{e}"

In [ ]:
def parse_action(text):
    """從模型輸出抓出 (工具名, 參數dict)；抓不到回 (None, None)。"""
    m_name = re.search(r"Action:\s*([A-Za-z_]\w*)", text)
    if not m_name:
        return None, None
    args = {}
    m_args = re.search(r"Action Input:\s*(\{.*\})", text, re.DOTALL)
    if m_args:
        try:
            args = json.loads(m_args.group(1))
        except Exception:
            args = {}
    return m_name.group(1), args

## 2. Tool Registry：工具即資料

每支工具登記 name / description / 函式。新增工具只要註冊一筆。

In [ ]:
def word_count(text: str) -> str:
    """回傳一段文字的字元數。"""
    return f"{len(text or '')} 個字元"


REGISTRY = [
    {"name": "get_current_time", "fn": lambda: get_current_time(),
     "description": "回傳現在台北時間，無參數。"},
    {"name": "calculator", "fn": calculator,
     "description": '計算算式，參數 {"expression": "3+4*2"}。'},
    {"name": "word_count", "fn": word_count,
     "description": '計算文字字元數，參數 {"text": "你好"}。'},
]
TOOLS = {t["name"]: t["fn"] for t in REGISTRY}

## 3. 從 registry 自動生成工具說明

新增工具，prompt 自己更新，不用改別處。

In [ ]:
def build_tools_prompt(registry):
    lines = [f"- {t['name']}：{t['description']}" for t in registry]
    return "\n".join(lines)


REACT_SYSTEM = f"""你是一個會使用工具、一步步推理的助手。可用工具：

{build_tools_prompt(REGISTRY)}

格式（一次一步）：
Thought: 推理
Action: 工具名稱
Action Input: JSON 參數

我會以 Observation 回覆。資訊足夠時改用：
Thought: 最後推理
Final Answer: 答案"""
print(REACT_SYSTEM)

## 4. 健壯的迴圈：把錯誤當 Observation 餵回

模型一定會偶爾**幻覺工具名、給錯參數**。健壯做法不是崩潰，而是把錯誤當成一種觀察餵回去，讓模型**下一步自我修正**。

In [ ]:
def route(name, args):
    """工具路由：把模型選的 name 對應到實際函式；出錯回錯誤訊息（當 Observation）。"""
    if name not in TOOLS:
        avail = ", ".join(TOOLS)
        return f"沒有名為 {name} 的工具。可用的是：{avail}"
    try:
        return str(TOOLS[name](**args))
    except TypeError as e:
        return f"參數錯誤：{e}"
    except Exception as e:
        return f"執行錯誤：{e}"


def run_agent(question, max_steps=6, verbose=True):
    scratchpad = ""
    for step in range(1, max_steps + 1):
        messages = [{"role": "system", "content": REACT_SYSTEM},
                    {"role": "user", "content": f"問題：{question}\n\n{scratchpad}"}]
        reply = chat(messages, max_new_tokens=256).split("Observation")[0].strip()
        if verbose:
            print(f"--- 第 {step} 步 ---\n{reply}")
        if "Final Answer:" in reply:
            return reply.split("Final Answer:")[-1].strip()
        name, args = parse_action(reply)
        obs = "（沒解析到 Action）" if name is None else route(name, args)
        if verbose:
            print(f"Observation: {obs}\n")
        scratchpad += f"{reply}\nObservation: {obs}\n"
    return "（達到最大步數仍未完成）"

## 5. 讓 agent 自己路由

In [ ]:
print(run_agent("『人工智慧』這四個字加起來有幾個字元，再乘以 3 是多少？"))

## 小結

- **Tool registry**：工具即資料，prompt 從 registry 自動生成，新增工具零改動。
- **工具路由**：把模型選的 name 對應到實際函式。
- **健壯性**：幻覺工具名 / 參數錯誤，都當成 **Observation 餵回**，讓模型自我修正。

下一課：給 agent **記憶**，讓它記得住對話、撐得久。